# 02 · Silver – Limpeza e padronização (Fabric)

Lê as tabelas Bronze e grava dados limpos, tipados e com nomes de negócio.

- **Colunas mapeadas pelo cabeçalho do SIDRA** (linha 0), não pela posição.
- **Códigos especiais do SIDRA** viram valor tratado + coluna `status_valor`: `-` = zero absoluto (vira 0); `..` = não se aplica, `...` = não disponível, `X` = sigilo (viram nulo).
- **Nada é descartado aqui.** A Silver só limpa e marca; quem decide o que fica de fora é a fase de qualidade e a Gold.
- **UF e região vêm do código IBGE** (2 primeiros dígitos = UF, primeiro dígito = região), não do texto do nome.

Saídas: `silver_populacao`, `silver_pib` e `silver_municipio`.

**Antes de rodar:** o `lake_visao` precisa estar anexado como lakehouse padrão.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# UFs pelo código IBGE (2 primeiros dígitos do código do município)
UFS = [
    ("11", "RO", "Rondônia"), ("12", "AC", "Acre"), ("13", "AM", "Amazonas"),
    ("14", "RR", "Roraima"), ("15", "PA", "Pará"), ("16", "AP", "Amapá"),
    ("17", "TO", "Tocantins"), ("21", "MA", "Maranhão"), ("22", "PI", "Piauí"),
    ("23", "CE", "Ceará"), ("24", "RN", "Rio Grande do Norte"), ("25", "PB", "Paraíba"),
    ("26", "PE", "Pernambuco"), ("27", "AL", "Alagoas"), ("28", "SE", "Sergipe"),
    ("29", "BA", "Bahia"), ("31", "MG", "Minas Gerais"), ("32", "ES", "Espírito Santo"),
    ("33", "RJ", "Rio de Janeiro"), ("35", "SP", "São Paulo"), ("41", "PR", "Paraná"),
    ("42", "SC", "Santa Catarina"), ("43", "RS", "Rio Grande do Sul"),
    ("50", "MS", "Mato Grosso do Sul"), ("51", "MT", "Mato Grosso"),
    ("52", "GO", "Goiás"), ("53", "DF", "Distrito Federal"),
]
REGIOES = [("1", "Norte"), ("2", "Nordeste"), ("3", "Sudeste"), ("4", "Sul"), ("5", "Centro-Oeste")]

df_uf = spark.createDataFrame(UFS, ["cod_uf", "sigla_uf", "nome_uf"])
df_regiao = spark.createDataFrame(REGIOES, ["cod_regiao", "nome_regiao"])

def gravar(df, tabela):
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true").saveAsTable(tabela))
    print(f"{tabela}: {spark.table(tabela).count()} linhas")

In [ ]:
def ler_bronze(nome):
    """Lê a Bronze e renomeia as colunas usando o cabeçalho do SIDRA (linha 0)."""
    df = spark.table(f"bronze_{nome}")
    cabecalho = df.filter("_linha = 0").first().asDict()
    col = {descricao: coluna for coluna, descricao in cabecalho.items()
           if not coluna.startswith("_")}
    return (df.filter("_linha > 0").select(
        F.col(col["Município (Código)"]).alias("cod_municipio"),
        F.col(col["Município"]).alias("municipio_uf_fonte"),
        F.col(col["Ano (Código)"]).cast("int").alias("ano"),
        F.col(col["Variável (Código)"]).alias("cod_variavel"),
        F.col(col["Unidade de Medida"]).alias("unidade"),
        F.trim(F.col(col["Valor"])).alias("valor_fonte"),
        F.col("_ingerido_em"),
    ))

def tratar_valor(df, tipo):
    """Converte o valor e registra o significado dos códigos especiais do SIDRA."""
    v = F.col("valor_fonte")
    numerico = v.rlike(r"^-?[0-9]+(\.[0-9]+)?$")
    status = (F.when(numerico, "ok")
               .when(v == "-", "zero_absoluto")
               .when(v == "..", "nao_se_aplica")
               .when(v == "...", "nao_disponivel")
               .when(v == "X", "sigilo")
               .otherwise("invalido"))
    valor = (F.when(numerico, v.cast(tipo))
              .when(v == "-", F.lit(0).cast(tipo)))
    return df.withColumn("status_valor", status).withColumn("valor", valor)

In [ ]:
# População: pessoas, número inteiro
pop = tratar_valor(ler_bronze("populacao"), "bigint")
silver_populacao = pop.select(
    "cod_municipio", "ano",
    F.col("valor").alias("populacao"),
    "status_valor", "valor_fonte", "_ingerido_em",
)
gravar(silver_populacao, "silver_populacao")

# PIB: a fonte vem em mil reais; guardo as duas unidades com o nome explícito
pib = tratar_valor(ler_bronze("pib"), "decimal(18,3)")
silver_pib = pib.select(
    "cod_municipio", "ano",
    F.col("valor").alias("pib_mil_reais"),
    (F.col("valor") * 1000).cast("decimal(21,3)").alias("pib_reais"),
    "unidade", "status_valor", "valor_fonte", "_ingerido_em",
)
gravar(silver_pib, "silver_pib")

In [ ]:
# Município: união das duas fontes, nome do ano mais recente
municipios = (
    pop.select("cod_municipio", "municipio_uf_fonte", "ano")
       .unionByName(pib.select("cod_municipio", "municipio_uf_fonte", "ano"))
)
mais_recente = Window.partitionBy("cod_municipio").orderBy(F.col("ano").desc())

silver_municipio = (
    municipios
    .withColumn("rn", F.row_number().over(mais_recente))
    .filter("rn = 1")
    .withColumn("nome_municipio", F.regexp_extract("municipio_uf_fonte", r"^(.*) - [A-Z]{2}$", 1))
    .withColumn("sigla_uf_nome", F.regexp_extract("municipio_uf_fonte", r" - ([A-Z]{2})$", 1))
    .withColumn("cod_uf", F.substring("cod_municipio", 1, 2))
    .withColumn("cod_regiao", F.substring("cod_municipio", 1, 1))
    .join(df_uf, "cod_uf", "left")
    .join(df_regiao, "cod_regiao", "left")
    .select(
        "cod_municipio", "nome_municipio", "cod_uf", "sigla_uf", "nome_uf",
        "cod_regiao", "nome_regiao",
        "sigla_uf_nome",            # UF lida do texto, para conferir na fase de qualidade
        F.col("ano").alias("ultimo_ano_na_fonte"),
    )
)
gravar(silver_municipio, "silver_municipio")

In [ ]:
# Conferência rápida (as validações completas ficam no notebook de qualidade)
print("== status dos valores")
for t in ["silver_populacao", "silver_pib"]:
    spark.table(t).groupBy("status_valor").count().show()

print("== municípios por região")
spark.table("silver_municipio").groupBy("nome_regiao").count().orderBy("nome_regiao").show()

print("== municípios que estão na população e não no PIB")
(spark.table("silver_populacao").select("cod_municipio").distinct()
   .subtract(spark.table("silver_pib").select("cod_municipio").distinct())
   .join(spark.table("silver_municipio"), "cod_municipio")
   .select("cod_municipio", "nome_municipio", "sigla_uf").show(truncate=False))